Notebook 9: Dataset 3 Independent Test + Grad-CAM
============================================================
Sequential Transfer Learning for Medicinal Plant Identification
Student: Hari (3146279) | BSc Software Engineering | University of Stirling

This notebook:
1. Evaluates the best model on Dataset 3 (completely held-out test set)
2. Generates Grad-CAM visualisations for selected species
3. Produces confusion matrix and per-species metrics

This tests generalisation — we need to verify
performance on data the model has NEVER seen during training.

Run in Google Colab with GPU runtime!
Runtime -> Change runtime type -> GPU

## Mount Drive and Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, f1_score
)
import seaborn as sns

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU: {gpus[0].name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

## Configuration

In [ ]:
IMG_SIZE = (224, 224)  # Must match training pipeline
BATCH_SIZE = 32
SEED = 42

DRIVE_BASE = "/content/drive/MyDrive"
PROJECT_DIR = os.path.join(DRIVE_BASE, "MedicinalPlant_Dissertation")
DATA_DIR = os.path.join(PROJECT_DIR, "data")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")
RESULTS_DIR = os.path.join(PROJECT_DIR, "dataset3_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Results will be saved to: {RESULTS_DIR}")

## Find and Load Dataset 3 (Independent Test Set)

In [ ]:
def find_image_directory(base_dir):
    """Walk directory tree to find folder containing class subdirectories with images."""
    if not os.path.exists(base_dir):
        return None
    for root, dirs, files in os.walk(base_dir):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        if not dirs:
            continue
        sample_dir = os.path.join(root, dirs[0])
        if os.path.isdir(sample_dir):
            sample_files = os.listdir(sample_dir)
            has_images = any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in sample_files)
            if has_images:
                return root
    return None

print(">>> Finding Dataset 3 (Independent Test Set)...")

dataset3_path = None
possible_paths = [
    os.path.join(DATA_DIR, "dataset3"),
    os.path.join(DATA_DIR, "Dataset 3"),
    os.path.join(DATA_DIR, "medicinal_leaf"),
]

for p in possible_paths:
    if os.path.exists(p):
        found = find_image_directory(p)
        if found:
            dataset3_path = found
            break

if dataset3_path is None:
    raise FileNotFoundError(
        "Could not find Dataset 3!\n"
        f"Expected at: {DATA_DIR}/dataset3/\n"
        "Make sure your data is in Google Drive.\n"
        "Dataset 3 = 'Medicinal Leaf Dataset' from Mendeley (~900 images, 9 species)"
    )

# List classes
d3_classes = sorted([
    d for d in os.listdir(dataset3_path)
    if os.path.isdir(os.path.join(dataset3_path, d)) and not d.startswith('.')
])
print(f"\nDataset 3 found: {dataset3_path}")
print(f"Classes ({len(d3_classes)}): {d3_classes}")

total_imgs = 0
for c in d3_classes:
    n = len([f for f in os.listdir(os.path.join(dataset3_path, c))
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    total_imgs += n
    print(f"  {c:30s} {n:4d} images")
print(f"  {'TOTAL':30s} {total_imgs:4d} images")

## Load the trained model

In [ ]:
print("\n>>> Loading trained model...")

model = None
model_paths = [
    os.path.join(MODEL_DIR, "phase3b_commercial.keras"),
    os.path.join(MODEL_DIR, "phase3b_best.keras"),
    os.path.join(MODEL_DIR, "phase3b_model.keras"),
    os.path.join(PROJECT_DIR, "phase3b_commercial.keras"),
]

for mp in model_paths:
    if os.path.exists(mp):
        print(f"  Trying: {mp}")
        try:
            model = keras.models.load_model(mp)
            print(f"  Loaded successfully!")
            break
        except Exception as e:
            print(f"  Failed: {e}")

if model is None:
    # Try loading from .h5 if .keras fails
    for ext in ['.h5']:
        h5_paths = [p.replace('.keras', ext) for p in model_paths]
        for mp in h5_paths:
            if os.path.exists(mp):
                try:
                    model = keras.models.load_model(mp)
                    print(f"  Loaded from {mp}")
                    break
                except Exception:
                    pass
        if model:
            break

if model is None:
    raise FileNotFoundError("Could not load any model! Check MODEL_DIR.")

# Load class names from training
class_names_path = None
for p in [
    os.path.join(MODEL_DIR, "class_names.json"),
    os.path.join(PROJECT_DIR, "class_names.json"),
]:
    if os.path.exists(p):
        class_names_path = p
        break

if class_names_path:
    with open(class_names_path, 'r') as f:
        train_class_names = json.load(f)
    print(f"Training classes ({len(train_class_names)}): {train_class_names}")
else:
    print("WARNING: class_names.json not found. Using model output layer size.")
    train_class_names = [f"class_{i}" for i in range(model.output_shape[-1])]

## Prepare Dataset 3 images for prediction

In [ ]:
# Since Dataset 3 may have different species names than the training set,
# we need to handle the mapping carefully.
print("\n>>> Preparing Dataset 3 for evaluation...")

# Load all images from Dataset 3
all_images = []
all_true_labels = []
all_filenames = []

for species_name in d3_classes:
    species_dir = os.path.join(dataset3_path, species_name)
    for img_file in os.listdir(species_dir):
        if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        img_path = os.path.join(species_dir, img_file)
        try:
            img = keras.utils.load_img(img_path, target_size=IMG_SIZE)
            img_array = keras.utils.img_to_array(img) / 255.0  # Normalise to [0, 1]
            all_images.append(img_array)
            all_true_labels.append(species_name)
            all_filenames.append(img_path)
        except Exception as e:
            print(f"  Skipping {img_path}: {e}")

all_images = np.array(all_images)
print(f"Loaded {len(all_images)} images from {len(d3_classes)} species")

## Run predictions on ALL Dataset 3 images

In [ ]:
print("\n>>> Running predictions on Dataset 3...")

predictions = model.predict(all_images, batch_size=BATCH_SIZE, verbose=1)
pred_class_indices = np.argmax(predictions, axis=1)
pred_confidences = np.max(predictions, axis=1)
pred_class_names = [train_class_names[i] for i in pred_class_indices]

# Summary statistics
print(f"\nPrediction Summary:")
print(f"  Total images: {len(all_images)}")
print(f"  Mean confidence: {np.mean(pred_confidences):.4f}")
print(f"  Median confidence: {np.median(pred_confidences):.4f}")
print(f"  Min confidence: {np.min(pred_confidences):.4f}")
print(f"  Max confidence: {np.max(pred_confidences):.4f}")

# Confidence distribution
low_conf = np.sum(pred_confidences < 0.5)
print(f"  Low confidence (<50%): {low_conf} ({low_conf/len(pred_confidences)*100:.1f}%)")

## Analysis: Species Overlap

In [ ]:
# Check which D3 species overlap with training species (case-insensitive)
print("\n>>> Analysing species overlap between Dataset 3 and training set...")

train_lower = {s.lower().strip(): s for s in train_class_names}
d3_lower = {s.lower().strip(): s for s in d3_classes}

overlap = set(train_lower.keys()) & set(d3_lower.keys())
d3_only = set(d3_lower.keys()) - set(train_lower.keys())

print(f"\nOverlapping species ({len(overlap)}):")
for s in sorted(overlap):
    print(f"  Training: {train_lower[s]:30s} ← → D3: {d3_lower[s]}")

print(f"\nDataset 3 only ({len(d3_only)}) - model has NEVER seen these:")
for s in sorted(d3_only):
    print(f"  {d3_lower[s]}")

# This is the KEY result: how does the model perform on species it was
# trained on vs species it has never seen?

# Separate predictions by overlap status
overlap_correct = 0
overlap_total = 0
novel_total = 0

for true_label, pred_label, conf in zip(all_true_labels, pred_class_names, pred_confidences):
    if true_label.lower().strip() in overlap:
        overlap_total += 1
        if pred_label.lower().strip() == true_label.lower().strip():
            overlap_correct += 1
    else:
        novel_total += 1

if overlap_total > 0:
    print(f"\nOverlapping species accuracy: {overlap_correct}/{overlap_total} = {overlap_correct/overlap_total*100:.2f}%")
print(f"Novel species (unseen): {novel_total} images")
print(f"  (Model cannot correctly classify these — zero-shot scenario)")

## Detailed Per-Species Results

In [ ]:
print("\n>>> Per-species prediction analysis on Dataset 3...")

from collections import Counter

for species in d3_classes:
    species_mask = [t == species for t in all_true_labels]
    species_preds = [pred_class_names[i] for i, m in enumerate(species_mask) if m]
    species_confs = [pred_confidences[i] for i, m in enumerate(species_mask) if m]

    pred_counts = Counter(species_preds)
    top3 = pred_counts.most_common(3)

    print(f"\n{species} ({len(species_preds)} images):")
    print(f"  Avg confidence: {np.mean(species_confs):.4f}")
    print(f"  Top predictions:")
    for pred, count in top3:
        pct = count / len(species_preds) * 100
        print(f"    {pred:35s} {count:3d} ({pct:5.1f}%)")

## Generate Confusion Matrix

In [ ]:
print("\n>>> Generating confusion matrix...")

# For the confusion matrix, we show what the model predicted for each D3 species
# This is an open-set recognition scenario for novel species

# Create a mapping plot showing D3 species → predicted taxa
fig, ax = plt.subplots(figsize=(14, 8))

# Build confusion-like matrix: rows = true D3 species, cols = predicted species
all_pred_species = sorted(set(pred_class_names))
matrix = np.zeros((len(d3_classes), len(all_pred_species)))

for true_label, pred_label in zip(all_true_labels, pred_class_names):
    row = d3_classes.index(true_label)
    col = all_pred_species.index(pred_label)
    matrix[row, col] += 1

# Normalise by row
row_sums = matrix.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
matrix_norm = matrix / row_sums

# Only show columns that have any predictions
col_mask = matrix.sum(axis=0) > 0
shown_cols = [c for c, m in zip(all_pred_species, col_mask) if m]
matrix_shown = matrix_norm[:, col_mask]

sns.heatmap(matrix_shown, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=shown_cols, yticklabels=d3_classes,
            ax=ax, vmin=0, vmax=1, linewidths=0.5)
ax.set_xlabel('Predicted Species (from training set)')
ax.set_ylabel('True Species (Dataset 3)')
ax.set_title('Dataset 3 Independent Test: Prediction Distribution\n(Normalised by true species)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

fig.savefig(os.path.join(RESULTS_DIR, 'dataset3_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR}/dataset3_confusion_matrix.png")

## Confidence Distribution Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall confidence histogram
axes[0].hist(pred_confidences, bins=50, color='#3498db', alpha=0.7, edgecolor='white')
axes[0].axvline(x=0.5, color='red', linestyle='--', label='50% threshold')
axes[0].set_xlabel('Prediction Confidence')
axes[0].set_ylabel('Count')
axes[0].set_title('Confidence Distribution (All Dataset 3 Predictions)')
axes[0].legend()

# Per-species mean confidence
species_mean_conf = []
for species in d3_classes:
    mask = [t == species for t in all_true_labels]
    confs = [pred_confidences[i] for i, m in enumerate(mask) if m]
    species_mean_conf.append(np.mean(confs))

colors = ['#2ecc71' if species.lower().strip() in overlap else '#e74c3c' for species in d3_classes]
bars = axes[1].barh(d3_classes, species_mean_conf, color=colors, alpha=0.7)
axes[1].set_xlabel('Mean Confidence')
axes[1].set_title('Mean Prediction Confidence per Species\n(Green=seen during training, Red=novel)')
axes[1].set_xlim(0, 1)
axes[1].axvline(x=0.5, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'dataset3_confidence.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR}/dataset3_confidence.png")

## Grad-CAM Visualisations

In [ ]:
print("\n>>> Generating Grad-CAM visualisations...")

def make_gradcam_heatmap(model, img_array, pred_index=None):
    """Generate Grad-CAM heatmap (Keras 3 / TF 2.16+ compatible)."""
    # Find the last convolutional layer
    last_conv_layer = None
    for layer in reversed(model.layers):
        # Check inside nested models (e.g., ResNet50 backbone)
        if hasattr(layer, 'layers'):
            for sublayer in reversed(layer.layers):
                if isinstance(sublayer, tf.keras.layers.Conv2D):
                    last_conv_layer = sublayer
                    break
            if last_conv_layer:
                break
        # Check top-level conv layers
        if isinstance(layer, tf.keras.layers.Conv2D):
            last_conv_layer = layer
            break
        # Also match layers with 4D output (backbone model output)
        try:
            if hasattr(layer, 'output') and len(layer.output.shape) == 4:
                last_conv_layer = layer
                break
        except (AttributeError, RuntimeError):
            continue

    if last_conv_layer is None:
        print("  WARNING: Could not find convolutional layer for Grad-CAM")
        return None

    # Build gradient model
    # Use model.layers[-1].output instead of model.output for Keras 3
    try:
        grad_model = tf.keras.Model(
            model.inputs,
            [last_conv_layer.output, model.layers[-1].output]
        )
    except (AttributeError, ValueError) as e:
        print(f"  Grad-CAM model build failed: {e}")
        return None

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    if grads is None:
        return None

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_heatmap(img, heatmap, alpha=0.4):
    """Overlay Grad-CAM heatmap on the original image."""
    import cv2
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_colored = plt.cm.jet(heatmap_resized)[:, :, :3]
    overlaid = img * (1 - alpha) + heatmap_colored * alpha
    return np.clip(overlaid, 0, 1)


# Select representative images: 1 per species
print("Generating Grad-CAM for selected images...")
fig_rows = len(d3_classes)
fig, axes = plt.subplots(fig_rows, 3, figsize=(12, 4 * fig_rows))

for row, species in enumerate(d3_classes):
    # Find first image of this species
    species_indices = [i for i, t in enumerate(all_true_labels) if t == species]
    if not species_indices:
        continue

    idx = species_indices[0]
    img = all_images[idx]
    img_batch = np.expand_dims(img, 0)

    pred_idx = pred_class_indices[idx]
    pred_name = pred_class_names[idx]
    pred_conf = pred_confidences[idx]

    # Original image
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f'True: {species}', fontsize=9)
    axes[row, 0].axis('off')

    # Grad-CAM
    heatmap = make_gradcam_heatmap(model, img_batch, pred_idx)
    if heatmap is not None:
        axes[row, 1].imshow(heatmap, cmap='jet')
        axes[row, 1].set_title('Grad-CAM Heatmap', fontsize=9)
    else:
        axes[row, 1].text(0.5, 0.5, 'N/A', ha='center', va='center')
        axes[row, 1].set_title('Grad-CAM (failed)', fontsize=9)
    axes[row, 1].axis('off')

    # Overlay
    if heatmap is not None:
        overlay = overlay_heatmap(img, heatmap)
        axes[row, 2].imshow(overlay)
    else:
        axes[row, 2].imshow(img)
    axes[row, 2].set_title(f'Pred: {pred_name} ({pred_conf:.1%})', fontsize=9)
    axes[row, 2].axis('off')

fig.suptitle('Grad-CAM Visualisations: Dataset 3 (Independent Test Set)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'gradcam_dataset3.png'), dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR}/gradcam_dataset3.png")

## Save Complete Results

In [ ]:
print("\n>>> Saving complete results...")

results = {
    "dataset": "Dataset 3 (Independent Test Set)",
    "total_images": len(all_images),
    "species_count": len(d3_classes),
    "species": d3_classes,
    "overlapping_species": [d3_lower[s] for s in sorted(overlap)],
    "novel_species": [d3_lower[s] for s in sorted(d3_only)],
    "mean_confidence": float(np.mean(pred_confidences)),
    "median_confidence": float(np.median(pred_confidences)),
    "low_confidence_pct": float(low_conf / len(pred_confidences) * 100),
}

if overlap_total > 0:
    results["overlapping_species_accuracy"] = float(overlap_correct / overlap_total)

# Per-species breakdown
per_species = {}
for species in d3_classes:
    mask = [t == species for t in all_true_labels]
    preds = [pred_class_names[i] for i, m in enumerate(mask) if m]
    confs = [float(pred_confidences[i]) for i, m in enumerate(mask) if m]
    per_species[species] = {
        "count": sum(mask),
        "mean_confidence": float(np.mean(confs)),
        "top_prediction": Counter(preds).most_common(1)[0] if preds else None,
        "is_novel": species.lower().strip() not in overlap,
    }
results["per_species"] = per_species

with open(os.path.join(RESULTS_DIR, 'dataset3_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nAll results saved to: {RESULTS_DIR}/")
print(f"Files:")
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f"  {f}")

print("\n" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)
print(f"\nKey results:")
if overlap_total > 0:
    print(f"  Overlapping species accuracy: {overlap_correct/overlap_total*100:.2f}%")
print(f"  Mean confidence: {np.mean(pred_confidences)*100:.2f}%")
print(f"  Novel species (unseen): {len(d3_only)} out of {len(d3_classes)}")
print(f"  Low confidence predictions (<50%): {low_conf/len(pred_confidences)*100:.1f}%")